In [1]:
# Training a Transformer Based Model for Emotion Classification
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
# Defining Model 
class EmotionTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, hidden_dim, num_layers, num_classes, max_len, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.pos_embedding = nn.Embedding(max_len , embed_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
        d_model = embed_dim,
        nhead= num_heads,
        dim_feedforward = hidden_dim,
        dropout = dropout,
        batch_first = True
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers = num_layers
        )
        
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self , x):
        batch_size , seq_len = x.shape

        positions = (torch.arange(seq_len , device = x.device).unsqueeze(0).expand(batch_size, seq_len))

        x = self.embedding(x) + self.pos_embedding(positions)
        x = self.transformer(x)
        x = x.mean(dim =1)
        x = self.dropout(x)
        return self.fc(x)


In [4]:
# Loading Vocabulary and Unique Emotion List 
import pickle 
base_url = './preProcessedInputs/'
vocabulary = {}
unique_emotion_list = []
max_len = 0

with open(f'{base_url}vocab.pkl', 'rb') as f:
    vocabulary = pickle.load(f)

with open(f'{base_url}unique_emotion_list.pkl' , 'rb' ) as f:
    unique_emotion_list = pickle.load(f)

with open(f'{base_url}max_len.pkl' , 'rb') as f:
    max_len = pickle.load(f)

# vocab_size = len(vocabulary)
vocab_size = max(vocabulary.values()) + 1
embed_dim = 128
hidden_dim = 256 
num_classes = len(unique_emotion_list)


In [5]:
# Model Initialization 
model = EmotionTransformer(
    vocab_size = vocab_size,
    embed_dim = embed_dim,
    num_heads = 8, 
    hidden_dim = hidden_dim,
    num_layers = 2,
    num_classes = num_classes,
    max_len = 60
)

In [7]:
# Loading DataSet 
from joblib import load 
data = load(f'{base_url}dataset.joblib')

X_train = data["X_train"]
y_train = data["y_train"]
X_test  = data["X_test"]
y_test  = data["y_test"]
# We have saved y as one hot encoding dataframe -> convert it numpy array

y_train = y_train.to_numpy()
y_test = y_test.to_numpy()
#  One-hot -> class indices
y_train = np.argmax(y_train, axis = 1)
y_test = np.argmax(y_test , axis = 1)

# Converting to pytorch Tensor 
X_train = torch.tensor(X_train ,dtype = torch.long)
y_train = torch.tensor(y_train,dtype = torch.long)
X_test  = torch.tensor(X_test,dtype = torch.long)
y_test  = torch.tensor(y_test,dtype = torch.long)

In [33]:

model.load_state_dict(torch.load("Transformer/transformer.pth"))

<All keys matched successfully>

In [34]:
# Model Training 

import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader


device = torch.device ("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device ("cpu")
print("Device is : " , device)
model = model.to(device)

# Loss + optimizer 
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

# Dataset + DataLoader 
train_dataset = TensorDataset(X_train , y_train)
train_loader = DataLoader(train_dataset , batch_size = 64, shuffle = True)

# Training Loop

epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch_X, batch_y in train_loader:
        # Move to Device 
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        # Forward Pass 
        
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)

        # Backprop
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss : {total_loss:.4f}")


Device is :  cuda
Epoch 1/5, Loss : 47.2936
Epoch 2/5, Loss : 46.6570
Epoch 3/5, Loss : 43.6950
Epoch 4/5, Loss : 42.6290
Epoch 5/5, Loss : 41.9091


In [35]:
torch.save(model.state_dict(), "Transformer/transformer.pth")
print("Model Saved Successfully")

Model Saved Successfully


In [21]:
# Using
import torch
import joblib
import numpy as np
import nltk

vocabulary = joblib.load("preProcessedInputs/vocab.pkl")
emotion_labels = joblib.load("preProcessedInputs/unique_emotion_list.pkl")
from sklearn.preprocessing import OneHotEncoder
def wordTokenize(text:str):
    return nltk.word_tokenize(text)
model = EmotionTransformer(
    vocab_size=len(vocabulary),
    embed_dim=128,
    num_heads=4,
    hidden_dim=256,
    num_layers=2,
    num_classes=len(emotion_labels),
    max_len=60
)

model.load_state_dict(
    torch.load("Transformer/transformer.pth", map_location="cpu")
)

model.eval()


def preprocess_text(text):

    text = text.lower().strip()

    tokens = wordTokenize(text)

    sequence = []

    for word in tokens:
        sequence.append(
            vocabulary.get(word, vocabulary["<UNK>"])
        )

    if len(sequence) < 60:
        sequence += [0] * (60 - len(sequence))
    else:
        sequence = sequence[:60]

    return np.array(sequence)


def predict_emotion(text):

    sequence = preprocess_text(text)

    x = torch.tensor(
        sequence,
        dtype=torch.long
    ).unsqueeze(0)

    with torch.no_grad():

        logits = model(x)

        probabilities = torch.softmax(
            logits,
            dim=1
        )

        pred_idx = torch.argmax(
            probabilities,
            dim=1
        ).item()

        confidence = probabilities[
            0,
            pred_idx
        ].item()

    return {
        "emotion": emotion_labels[pred_idx],
        "confidence": round(confidence * 100, 2)
    }



In [25]:
print(predict_emotion(input("Enter your Sentence")))

Enter your Sentence I am feeling extremely annoyed


{'emotion': np.str_('neutral'), 'confidence': 100.0}


In [36]:
import torch
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from torch.utils.data import DataLoader, TensorDataset

# 1. Setup device and ready the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

# 2. Dataset and Loader
# CRITICAL: Ensure X_test is a LongTensor of token IDs, padded/truncated to your max_len!
# Shape must be: (num_samples, max_len)
if not isinstance(X_test, torch.Tensor):
    X_test = torch.tensor(X_test, dtype=torch.long)
if not isinstance(y_test, torch.Tensor):
    y_test = torch.tensor(y_test, dtype=torch.long)

test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

all_preds = []
all_labels = []

# 3. Evaluation Loop
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        # Move inputs to the correct device
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        # Forward pass (Outputs shape: [batch_size, num_classes])
        logits = model(batch_X)
        
        # Get the predicted class indices along dimension 1
        _, predicted = torch.max(logits, dim=1)

        # Move to CPU and store for sklearn metrics
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(batch_y.cpu().numpy())

# 4. Compute Metrics
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average="weighted", zero_division=0)
recall = recall_score(all_labels, all_preds, average="weighted", zero_division=0)
f1 = f1_score(all_labels, all_preds, average="weighted", zero_division=0)

# 5. Display Results
print("=== Transformer Performance ===")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

=== Transformer Performance ===
Accuracy  : 0.9981
Precision : 0.9982
Recall    : 0.9981
F1 Score  : 0.9981
